# Enterprise Surveys country dashboard

This notebook builds **one web page** (`index.html`): an interactive dashboard of the World Bank
Enterprise Surveys indicators. On the page you pick a country, a topic and what to compare with
(other countries, the region, the income group, all countries). The charts, the table and the CSV
download change with your choices.

How it works:
1. It finds every national Enterprise Surveys indicator in the World Bank Data360 API (about 200).
2. It collects each indicator for all countries and keeps the latest and the previous survey.
3. It checks the data, then writes everything into a single file, `index.html`.

The page needs no server and no API key: the data sits inside the file. To refresh the dashboard,
run this notebook again and replace `index.html`.

**To run:** click **Runtime > Run all**. It takes about a minute. `index.html` then downloads.

Only public data is used. This is a personal project and not an official World Bank product.

In [ ]:
# ---- 1. Settings --------------------------------------------------------------------
DEFAULT_COUNTRY = "ETH"   # the country the page opens on (three-letter ISO code)
FIRST_YEAR = 2010         # ignore surveys older than this

In [ ]:
# ---- 2. Find and collect every Enterprise Surveys indicator -------------------------
# The functions in this cell are the same as in my country brief notebook
# (github.com/bezaafework2-dotcom/enterprise-surveys-country-briefs).
import datetime
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests

DATA360 = "https://data360api.worldbank.org/data360"     # indicators and data
WB_API = "https://api.worldbank.org/v2"                  # country list (region, income group)
DEFAULT_COUNTRY = DEFAULT_COUNTRY.strip().upper()

# Topic of each indicator, as listed in the Enterprise Surveys "Indicator Descriptions" document
# (26 October 2026, enterprisesurveys.org). The API has no topic field, so the topic is read from the
# code: WB_ES_T_FIN14 belongs to the family FIN. A few codes sit under a different topic from the rest of
# their family; those are listed one by one.
TOPIC_OF_FAMILY = {
    "REG": "Regulations", "BUS": "Regulations", "TAX": "Taxes", "FIN": "Finance",
    "IN": "Infrastructure and Climate", "GEND": "Gender", "TR": "Trade",
    "COMP": "Competition and Public Procurement", "CORR": "Corruption", "GRAFT": "Corruption",
    "INFOR": "Informality", "DISP": "Dispute Resolution", "CRIME": "Crime", "JOBS": "Jobs",
    "WK": "Workforce", "MGMT": "Management Practices", "T": "Innovation and Technology",
    "CAR": "Firm Profile", "LFORM": "Firm Profile", "PERF": "Firm Profile", "OBST": "Biggest Obstacle",
}
TOPIC_OF_CODE = {
    "REG2": "Taxes", "REG4": "Taxes", "REG5": "Taxes", "REG6": "Taxes",     # tax visits, tax constraints
    "T2": "Finance",                                                        # audited financial statement
    "T3": "Firm Profile",                                                   # capacity utilization
    "CRIME9": "Dispute Resolution", "CORR12": "Dispute Resolution",         # courts
}


def topic_of(code):
    """Topic of an indicator code such as WB_ES_T_BREADY_TAX4_MEDIAN (its core is TAX4)."""
    core = code.replace("WB_ES_T_", "").replace("BREADY_", "").split("_")[0]
    return TOPIC_OF_CODE.get(core, TOPIC_OF_FAMILY.get(re.match(r"[A-Z]*", core).group(), "Other"))


UNITS = {"Percentage": "%", "Percentage of firms": "%", "Days": "days", "Hours": "hours",
         "Weeks": "weeks", "Months": "months", "Years": "years"}


def call_api(url, params=None, body=None):
    """One call to a World Bank API (GET, or POST when a body is given), tried up to 4 times
    because the APIs are sometimes slow. Returns the decoded JSON."""
    waits = [5, 20, 45]                       # seconds to wait before the 2nd, 3rd and 4th try
    for attempt in range(len(waits) + 1):
        try:
            if body is None:
                r = requests.get(url, params=params, timeout=120)
            else:
                r = requests.post(url, json=body, timeout=120)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as err:
            if attempt == len(waits):
                raise RuntimeError("A World Bank API did not answer after 4 tries. It is probably busy: "
                                   "wait a few minutes, then click Runtime > Run all again.") from err
            print(f"    API slow ({type(err).__name__}); trying again in {waits[attempt]} seconds ...")
            time.sleep(waits[attempt])


def find_indicators():
    """Every national-level Enterprise Surveys indicator in Data360, with its name, unit and topic.
    Codes starting WB_ES_T_ cover all firms; WB_ES_M_ and WB_ES_S_ repeat them by sector."""
    answer = call_api(f"{DATA360}/searchv2", body={
        "count": True, "search": "*", "top": 1000, "skip": 0,
        "select": "series_description/idno, series_description/name, series_description/measurement_unit",
        "filter": "series_description/database_id eq 'WB_ES'"})
    rows = []
    for item in answer.get("value", []):
        d = item.get("series_description") or {}
        code = d.get("idno") or ""
        if not code.startswith("WB_ES_T_"):
            continue
        rows.append({"code": code, "short_code": code.replace("WB_ES_T_", ""),
                     "indicator": (d.get("name") or code).strip(),
                     "unit": UNITS.get(d.get("measurement_unit"), ""),
                     "topic": topic_of(code)})
    return pd.DataFrame(rows).sort_values("code").reset_index(drop=True)


def get_indicator(code):
    """National values for one indicator: all countries, all survey years.
    The three breakdown filters keep the all-firms figure and drop the size and sector splits."""
    rows, skip = [], 0
    while True:
        answer = call_api(f"{DATA360}/data", params={
            "DATABASE_ID": "WB_ES", "INDICATOR": code, "COMP_BREAKDOWN_1": "_T",
            "COMP_BREAKDOWN_2": "_T", "COMP_BREAKDOWN_3": "_T", "skip": skip})
        page = answer.get("value") or []
        rows += [{"code": code, "iso3": r["REF_AREA"], "year": int(float(r["TIME_PERIOD"])),
                  "value": float(r["OBS_VALUE"])}
                 for r in page if r.get("OBS_VALUE") not in (None, "")]
        skip += len(page)
        if not page or skip >= int(answer.get("count") or 0):
            return rows


def get_countries():
    """Country list with region and income group. Regional aggregates are dropped."""
    records = call_api(f"{WB_API}/country", params={"format": "json", "per_page": 400})[1]
    df = pd.DataFrame([
        {"iso3": r["id"], "country": r["name"], "region": r["region"]["value"].strip(),
         "income": r["incomeLevel"]["value"].strip()}
        for r in records
    ])
    return df[df["region"] != "Aggregates"].reset_index(drop=True)


print("Finding the Enterprise Surveys indicators ...")
indicators = find_indicators()
assert len(indicators) > 0, "The indicator list came back empty."
print(f"Found {len(indicators)} indicators in {indicators['topic'].nunique()} topics.")

print(f"Collecting {len(indicators)} indicators for all countries (about a minute) ...")
with ThreadPoolExecutor(max_workers=4) as pool:                 # four calls at a time
    pages = list(pool.map(get_indicator, indicators["code"]))
raw = pd.DataFrame([row for page in pages for row in page])
raw = raw[raw["year"] >= FIRST_YEAR]
countries = get_countries()
raw = raw.merge(countries, on="iso3", how="inner")
print(f"Collected {len(raw):,} values for {raw['iso3'].nunique()} countries.")

In [ ]:
# ---- 3. Check the data, then keep the latest and the previous survey ----------------
# Rule-based checks. The notebook stops with a message if one fails, so a broken page is never written.
assert not raw.duplicated(["code", "iso3", "year"]).any(), "More than one value for a country, indicator and year."
assert raw["value"].notna().all(), "There are missing values."
assert raw["year"].between(FIRST_YEAR, datetime.date.today().year).all(), "A survey year is outside the expected range."
assert DEFAULT_COUNTRY in set(raw["iso3"]), f"No Enterprise Surveys indicators found for '{DEFAULT_COUNTRY}'."

# Shares of firms must lie between 0 and 100. Growth rates and similar can be negative, so only
# indicators named "Percent of firms ..." are checked. Anything outside is listed, not hidden.
shares = indicators[indicators["indicator"].str.lower().str.startswith("percent of firms")]["code"]
outside = raw[raw["code"].isin(shares) & ~raw["value"].between(-0.01, 100.01)]
print(f"Check: {len(outside)} 'percent of firms' values fall outside 0 to 100.")
if len(outside):
    print(outside[["code", "iso3", "year", "value"]].to_string(index=False))

raw = raw.sort_values(["code", "iso3", "year"])
raw["back"] = raw.groupby(["code", "iso3"]).cumcount(ascending=False)     # 0 = latest survey, 1 = the one before
latest = raw[raw["back"] == 0].set_index(["code", "iso3"])
previous = raw[raw["back"] == 1].set_index(["code", "iso3"])

places = raw.drop_duplicates("iso3").sort_values("country")               # countries that have data
indicators = indicators[indicators["code"].isin(set(raw["code"]))].reset_index(drop=True)
print(f"Kept {len(indicators)} indicators with data and {len(places)} countries.")

In [ ]:
# ---- 4. Put the data in the shape the page reads -------------------------------------
# One list per indicator, in the same order as the country list:
#   v = latest value, y = its survey year, pv = previous value, py = its survey year (None = no survey).
order = list(places["iso3"])


def wide(table, field, digits=None):
    """One list per indicator code: the field for every country, in the order of the country list."""
    grid = table[field].unstack("iso3").reindex(index=indicators["code"], columns=order)
    return {code: [None if pd.isna(x) else (round(float(x), digits) if digits else int(x)) for x in row]
            for code, row in zip(grid.index, grid.to_numpy())}


v, y = wide(latest, "value", 2), wide(latest, "year")
pv, py = wide(previous, "value", 2), wide(previous, "year")

DATA = {
    "generated": datetime.date.today().strftime("%d %B %Y"),
    "first_year": FIRST_YEAR,
    "default_country": DEFAULT_COUNTRY,
    "countries": [{"c": r.iso3, "n": r.country, "r": r.region, "i": r.income} for r in places.itertuples()],
    "indicators": [{"id": r.short_code, "n": r.indicator, "u": r.unit, "t": r.topic,
                    "v": v[r.code], "y": y[r.code], "pv": pv[r.code], "py": py[r.code]}
                   for r in indicators.itertuples()],
}

# Check the shape before writing: every list has one entry per country, and a value always has a year.
for ind in DATA["indicators"]:
    assert all(len(ind[k]) == len(order) for k in ("v", "y", "pv", "py")), ind["id"]
    assert all((v is None) == (y is None) for v, y in zip(ind["v"], ind["y"])), ind["id"]
    assert all(pv is None or v is not None for v, pv in zip(ind["v"], ind["pv"])), ind["id"]
print(f"Ready: {len(DATA['indicators'])} indicators x {len(order)} countries.")

In [ ]:
# ---- 5. The page: HTML, CSS and JavaScript ---------------------------------------------
# The dashboard is one self-contained page. __DATA__ is replaced by the data in the next cell.
# The JavaScript draws the charts and redraws them whenever a choice changes.
PAGE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Enterprise Surveys country dashboard</title>
<style>
  :root { --surface:#fcfcfb; --card:#ffffff; --ink:#0b0b0b; --ink2:#52514e; --muted:#898781; --line:#e1e0d9;
          --axis:#c3c2b7; --accent:#2a78d6; --accent-light:#a9c9ee; --other:#8f8e86; --group:#c3c2b7; }
  * { box-sizing: border-box; }
  body { margin:0; background:var(--surface); color:var(--ink); font:14px/1.45 -apple-system,"Segoe UI",Roboto,Arial,sans-serif; }
  main { max-width:1180px; margin:0 auto; padding:26px 22px 40px; }
  h1 { font-size:24px; margin:0 0 4px; }
  .sub { color:var(--ink2); margin:0 0 18px; max-width:860px; }
  .controls { display:flex; flex-wrap:wrap; gap:12px 16px; align-items:flex-end; padding:14px 16px; background:var(--card);
              border:1px solid var(--line); border-radius:8px; margin-bottom:14px; }
  .controls label { display:flex; flex-direction:column; gap:4px; font-size:12px; color:var(--ink2); }
  select, button { font:inherit; font-size:13px; padding:6px 8px; border:1px solid var(--axis); border-radius:5px; background:#fff; color:var(--ink); }
  select { min-width:150px; max-width:230px; }
  button { cursor:pointer; }
  button:hover, select:hover { border-color:var(--ink2); }
  .checks { display:flex; flex-wrap:wrap; gap:4px 14px; font-size:13px; color:var(--ink); }
  .checks label { flex-direction:row; align-items:center; gap:5px; font-size:13px; color:var(--ink); }
  .tiles { display:flex; flex-wrap:wrap; gap:10px; margin-bottom:14px; }
  .tile { background:var(--card); border:1px solid var(--line); border-radius:8px; padding:9px 14px; min-width:130px; }
  .tile b { display:block; font-size:19px; line-height:1.25; }
  .tile span { color:var(--ink2); font-size:12px; }
  .bar-row { display:flex; justify-content:space-between; align-items:center; margin:2px 0 12px; gap:12px; flex-wrap:wrap; }
  .legend { display:flex; flex-wrap:wrap; gap:6px 16px; font-size:12px; color:var(--ink2); }
  .legend i { display:inline-block; width:11px; height:11px; border-radius:2px; margin-right:5px; vertical-align:-1px; }
  .cards { display:grid; grid-template-columns:repeat(auto-fill,minmax(430px,1fr)); gap:12px; }
  .card { background:var(--card); border:1px solid var(--line); border-radius:8px; padding:12px 14px 10px; }
  .card .topic { font-size:10.5px; letter-spacing:.06em; text-transform:uppercase; color:var(--muted); }
  .card h2 { font-size:13.5px; font-weight:600; margin:2px 0 8px; line-height:1.35; }
  .row { display:grid; grid-template-columns:170px 1fr; align-items:center; gap:8px; margin:3px 0; font-size:12px; }
  .row .name { color:var(--ink2); text-align:right; line-height:1.2; overflow-wrap:anywhere; }
  .row.main .name { color:var(--ink); font-weight:600; }
  .track { position:relative; height:16px; }
  .track .zero { position:absolute; top:-2px; bottom:-2px; width:1px; background:var(--axis); }
  .track .bar { position:absolute; top:2px; height:12px; border-radius:2px; min-width:1px; }
  .track .val { position:absolute; top:-1px; font-size:12px; white-space:nowrap; color:var(--ink); }
  .row.main .val { font-weight:700; }
  .note { margin-top:7px; font-size:11.5px; color:var(--muted); }
  .flag { color:#8a5a00; }
  table { border-collapse:collapse; width:100%; background:var(--card); font-size:12.5px; }
  th, td { border-bottom:1px solid var(--line); padding:6px 8px; text-align:right; }
  th:first-child, td:first-child, th:nth-child(2), td:nth-child(2) { text-align:left; }
  th { background:#f1f0ea; position:sticky; top:0; font-weight:600; }
  .tablewrap { overflow:auto; border:1px solid var(--line); border-radius:8px; max-height:70vh; }
  .empty { padding:30px; text-align:center; color:var(--ink2); background:var(--card); border:1px solid var(--line); border-radius:8px; }
  footer { margin-top:22px; color:var(--muted); font-size:12px; max-width:900px; }
  footer a { color:var(--ink2); }
  @media (max-width:520px) { .cards { grid-template-columns:1fr; } .row { grid-template-columns:110px 1fr; } }
</style>
</head>
<body>
<main>
  <h1>Enterprise Surveys country dashboard</h1>
  <p class="sub">Pick a country, a topic and what to compare with. Each panel shows the country's latest survey, its previous survey,
     and the comparators you choose. Data: World Bank Enterprise Surveys, through the World Bank Data360 API.</p>

  <div class="controls">
    <label>Country <select id="country"></select></label>
    <label>Topic <select id="topic"></select></label>
    <label>Compare with <select id="c1"></select></label>
    <label>and <select id="c2"></select></label>
    <label>and <select id="c3"></select></label>
    <label>Group figure <select id="stat"><option value="average">Average</option><option value="median">Median</option></select></label>
    <div>
      <div style="font-size:12px;color:var(--ink2);margin-bottom:4px">Groups</div>
      <div class="checks">
        <label><input type="checkbox" id="g_region" checked> Region</label>
        <label><input type="checkbox" id="g_income" checked> Income group</label>
        <label><input type="checkbox" id="g_all"> All countries</label>
      </div>
    </div>
  </div>

  <div class="tiles" id="tiles"></div>
  <div class="bar-row">
    <div class="legend" id="legend"></div>
    <div><button id="view">Show as table</button> <button id="csv">Download CSV</button></div>
  </div>
  <div id="out"></div>

  <footer>
    <p id="source"></p>
    <p>Group figures are simple averages (or medians) of each country's most recent survey. They are calculated on this page and are not
       official World Bank aggregates. Countries are surveyed in different years, so comparisons mix survey years. "What stands out" ranks
       indicators by how far the country sits from the rest of its region and by how much it changed since its previous survey.
       This is a personal project and not an official World Bank product.
       Code and method: <a href="https://github.com/bezaafework2-dotcom/enterprise-surveys-dashboard">GitHub repository</a>.</p>
  </footer>
</main>

<script>
var DATA = __DATA__;

var $ = function (id) { return document.getElementById(id); };
var esc = function (s) { return String(s).replace(/[&<>"]/g, function (c) { return {'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]; }); };
var fmt = function (v, u) { return v === null || v === undefined ? '' : (u === '%' ? v.toFixed(1) + '%' : v.toFixed(1) + (u ? ' ' + u : '')); };
var STAND_OUT = 'What stands out';
var tableView = false;

function stat(values) {
  var v = values.filter(function (x) { return x !== null; });
  if (!v.length) return null;
  if ($('stat').value === 'median') {
    v.sort(function (a, b) { return a - b; });
    var m = Math.floor(v.length / 2);
    return v.length % 2 ? v[m] : (v[m - 1] + v[m]) / 2;
  }
  return v.reduce(function (a, b) { return a + b; }, 0) / v.length;
}

function members(field, value) {            // indexes of the countries in a group
  var out = [];
  DATA.countries.forEach(function (c, i) { if (field === null || c[field] === value) out.push(i); });
  return out;
}

function facts(ci) {                        // one record per indicator that has a value for the country
  var me = DATA.countries[ci], region = members('r', me.r);
  return DATA.indicators.map(function (ind) {
    var v = ind.v[ci];
    if (v === null) return null;
    var regionValues = region.map(function (i) { return ind.v[i]; }).filter(function (x) { return x !== null; });
    var rank = regionValues.filter(function (x) { return x > v; }).length + 1, n = regionValues.length;
    var position = n >= 5 ? (rank - 1) / (n - 1) : 0.5;
    var pv = ind.pv[ci], rel = pv === null ? 0 : Math.abs(v - pv) / Math.max(Math.abs(pv), 1);
    var bigChange = pv !== null && rel >= 0.3 && (ind.u !== '%' || Math.abs(v - pv) >= 5);
    return { ind: ind, v: v, y: ind.y[ci], pv: pv, py: ind.py[ci], rank: rank, n: n,
             signal: Math.abs(position - 0.5) * 2 + Math.min(rel, 1),
             high: n >= 5 && position <= 0.15, low: n >= 5 && position >= 0.85, bigChange: bigChange };
  }).filter(function (f) { return f !== null; });
}

function comparators(ci) {                  // [{label, kind, value(ind) -> number or null, year(ind)}]
  var me = DATA.countries[ci], out = [];
  ['c1', 'c2', 'c3'].forEach(function (id) {
    var k = $(id).value;
    if (k === '' || +k === ci) return;
    if (out.some(function (c) { return c.index === +k; })) return;
    out.push({ index: +k, kind: 'country', label: DATA.countries[+k].n,
               value: function (ind) { return ind.v[+k]; }, year: function (ind) { return ind.y[+k]; } });
  });
  var s = $('stat').value;
  [['g_region', 'r', me.r], ['g_income', 'i', me.i], ['g_all', null, 'All countries']].forEach(function (g) {
    if (!$(g[0]).checked) return;
    var idx = members(g[1], g[2]);
    out.push({ kind: 'group', label: g[2] + ' ' + s,
               value: function (ind) { return stat(idx.map(function (i) { return ind.v[i]; })); }, year: function () { return null; } });
  });
  return out;
}

function shown(ci) {
  var f = facts(ci), topic = $('topic').value;
  if (topic === STAND_OUT) return f.slice().sort(function (a, b) { return b.signal - a.signal; }).slice(0, 12);
  f = f.filter(function (x) { return x.ind.t === topic; });
  if (topic === 'Biggest Obstacle') return f.sort(function (a, b) { return b.v - a.v; });   // most-named obstacle first
  var key = function (x) { return x.ind.id.replace(/\d+/g, function (d) { return ('0000' + d).slice(-4); }); };
  return f.sort(function (a, b) { return key(a) < key(b) ? -1 : key(a) > key(b) ? 1 : 0; });  // in code order: FIN1, FIN2, ...
}

function card(f, me, comps) {
  var u = f.ind.u, rows = [{ label: me.n + ' ' + f.y, v: f.v, cls: 'main', color: 'var(--accent)' }];
  if (f.pv !== null) rows.push({ label: me.n + ' ' + f.py, v: f.pv, cls: '', color: 'var(--accent-light)' });
  comps.forEach(function (c) {
    var v = c.value(f.ind);
    if (v === null) return;
    rows.push({ label: c.label + (c.kind === 'country' ? ' ' + c.year(f.ind) : ''), v: v, cls: '',
                color: c.kind === 'country' ? 'var(--other)' : 'var(--group)' });
  });
  var lo = Math.min.apply(null, rows.map(function (r) { return r.v; }).concat([0]));
  var hi = Math.max.apply(null, rows.map(function (r) { return r.v; }).concat([0]));
  var span = (hi - lo) || 1, scale = 72 / span, zero = (0 - lo) * scale + (lo < 0 ? 14 : 0);   // % of the track; room for labels
  var html = rows.map(function (r) {
    var w = Math.abs(r.v) * scale, left = r.v >= 0 ? zero : zero - w;
    var label = r.v >= 0 ? 'left:calc(' + (left + w) + '% + 5px)' : 'right:calc(' + (100 - left) + '% + 5px)';
    return '<div class="row ' + r.cls + '"><div class="name" title="' + esc(r.label) + '">' + esc(r.label) + '</div>' +
           '<div class="track" title="' + esc(r.label + ': ' + fmt(r.v, u)) + '"><span class="zero" style="left:' + zero + '%"></span>' +
           '<span class="bar" style="left:' + left + '%;width:' + w + '%;background:' + r.color + '"></span>' +
           '<span class="val" style="' + label + '">' + fmt(r.v, u) + '</span></div></div>';
  }).join('');
  var notes = [];
  if (f.n >= 2) notes.push('Rank ' + f.rank + ' of ' + f.n + ' in ' + esc(me.r) + ' (1 = highest)');
  if (f.high) notes.push('among the highest in the region');
  if (f.low) notes.push('among the lowest in the region');
  if (f.bigChange) notes.push('<span class="flag">large change since ' + f.py + ': check the survey before using</span>');
  return '<div class="card"><div class="topic">' + esc(f.ind.t) + '</div><h2>' + esc(f.ind.n) + '</h2>' + html +
         '<div class="note">' + notes.join(' · ') + '</div></div>';
}

function tableRows(list, me, comps) {       // header + rows, used for the table view and the CSV
  var head = ['Topic', 'Indicator', 'Unit', me.n, 'Survey year', 'Previous survey', 'Previous year']
    .concat(comps.map(function (c) { return c.label; }));
  var rows = list.map(function (f) {
    return [f.ind.t, f.ind.n, f.ind.u, f.v, f.y, f.pv, f.py].concat(comps.map(function (c) { return c.value(f.ind); }));
  });
  return [head].concat(rows);
}

function render() {
  var ci = +$('country').value, me = DATA.countries[ci], comps = comparators(ci), list = shown(ci), all = facts(ci);
  var years = all.map(function (f) { return f.y; }), latest = Math.max.apply(null, years);
  var earlier = all.map(function (f) { return f.py; }).filter(function (y) { return y !== null; });
  $('tiles').innerHTML =
    '<div class="tile"><b>' + esc(me.n) + '</b><span>' + esc(me.r) + ' · ' + esc(me.i) + '</span></div>' +
    '<div class="tile"><b>' + latest + '</b><span>latest survey</span></div>' +
    '<div class="tile"><b>' + (earlier.length ? Math.max.apply(null, earlier) : 'None') + '</b><span>previous survey</span></div>' +
    '<div class="tile"><b>' + all.length + '</b><span>indicators with data</span></div>' +
    '<div class="tile"><b>' + list.length + '</b><span>shown now</span></div>';
  $('legend').innerHTML =
    '<span><i style="background:var(--accent)"></i>' + esc(me.n) + ', latest survey</span>' +
    '<span><i style="background:var(--accent-light)"></i>' + esc(me.n) + ', previous survey</span>' +
    (comps.some(function (c) { return c.kind === 'country'; }) ? '<span><i style="background:var(--other)"></i>Other countries</span>' : '') +
    (comps.some(function (c) { return c.kind === 'group'; }) ? '<span><i style="background:var(--group)"></i>Group ' + $('stat').value + 's</span>' : '');
  if (!list.length) {
    $('out').innerHTML = '<div class="empty">No indicators with data for ' + esc(me.n) + ' in this topic.</div>';
  } else if (tableView) {
    var t = tableRows(list, me, comps);
    $('out').innerHTML = '<div class="tablewrap"><table><thead><tr>' + t[0].map(function (h) { return '<th>' + esc(h) + '</th>'; }).join('') +
      '</tr></thead><tbody>' + t.slice(1).map(function (r) {
        return '<tr>' + r.map(function (c, j) {
          return '<td>' + (c === null ? '' : typeof c === 'number' && j !== 4 && j !== 6 ? c.toFixed(1) : esc(c)) + '</td>';
        }).join('') + '</tr>';
      }).join('') + '</tbody></table></div>';
  } else {
    $('out').innerHTML = '<div class="cards">' + list.map(function (f) { return card(f, me, comps); }).join('') + '</div>';
  }
  $('view').textContent = tableView ? 'Show as charts' : 'Show as table';
  try { history.replaceState(null, '', '#' + [me.c, encodeURIComponent($('topic').value)].join('/')); } catch (e) {}
}

function downloadCsv() {
  var ci = +$('country').value, me = DATA.countries[ci], t = tableRows(shown(ci), me, comparators(ci));
  var text = t.map(function (r) {
    return r.map(function (c) {
      var s = c === null ? '' : typeof c === 'number' ? String(Math.round(c * 10) / 10) : String(c);
      return /[",\n]/.test(s) ? '"' + s.replace(/"/g, '""') + '"' : s;
    }).join(',');
  }).join('\n');
  var a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob([text], { type: 'text/csv' }));
  a.download = 'enterprise_surveys_' + me.c + '.csv';
  document.body.appendChild(a); a.click(); a.remove();
}

function start() {
  var options = DATA.countries.map(function (c, i) { return { i: i, n: c.n }; }).sort(function (a, b) { return a.n.localeCompare(b.n); });
  $('country').innerHTML = options.map(function (o) { return '<option value="' + o.i + '">' + esc(o.n) + '</option>'; }).join('');
  ['c1', 'c2', 'c3'].forEach(function (id) {
    $(id).innerHTML = '<option value="">None</option>' + options.map(function (o) { return '<option value="' + o.i + '">' + esc(o.n) + '</option>'; }).join('');
  });
  var topics = [];
  DATA.indicators.forEach(function (ind) { if (topics.indexOf(ind.t) === -1) topics.push(ind.t); });
  topics.sort();
  $('topic').innerHTML = [STAND_OUT].concat(topics).map(function (t) { return '<option>' + esc(t) + '</option>'; }).join('');
  var wanted = (location.hash || '').replace('#', '').split('/');
  var first = DATA.countries.findIndex(function (c) { return c.c === (wanted[0] || DATA.default_country); });
  $('country').value = first >= 0 ? first : options[0].i;
  if (wanted[1]) { var t = decodeURIComponent(wanted[1]); if (topics.indexOf(t) !== -1) $('topic').value = t; }
  $('source').textContent = 'Source: World Bank Enterprise Surveys, World Bank Data360 API. Retrieved ' + DATA.generated + '. ' +
    DATA.indicators.length + ' indicators, ' + DATA.countries.length + ' countries, surveys since ' + DATA.first_year + '.';
  ['country', 'topic', 'c1', 'c2', 'c3', 'stat', 'g_region', 'g_income', 'g_all'].forEach(function (id) { $(id).addEventListener('change', render); });
  $('view').addEventListener('click', function () { tableView = !tableView; render(); });
  $('csv').addEventListener('click', downloadCsv);
  render();
}
start();
</script>
</body>
</html>
"""

In [ ]:
# ---- 6. Write index.html --------------------------------------------------------------
text = json.dumps(DATA, separators=(",", ":"), allow_nan=False, ensure_ascii=False).replace("</", "<\\/")
assert PAGE.count("__DATA__") == 1
page = PAGE.replace("__DATA__", text)
with open("index.html", "w", encoding="utf-8") as f:
    f.write(page)
print(f"Wrote index.html ({len(page.encode('utf-8')) / 1e6:.1f} MB).")

try:                                     # in Colab: download the file
    from google.colab import files
    files.download("index.html")
except ImportError:
    print("Open index.html in a browser to see the dashboard.")